Every non-complexion color across both collections, visualized by hue and lightness.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
SELECT
  era,
  product_type,
  product_name,
  shade_name,
  finish,
  c_pct,
  m_pct,
  y_pct,
  k_pct
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
WHERE product_type NOT IN ('Foundation', 'Concealer', 'Bronzer')
  AND c_pct IS NOT NULL
  AND m_pct IS NOT NULL
  AND y_pct IS NOT NULL
  AND k_pct IS NOT NULL
  AND shade_name IS NOT NULL
ORDER BY era, product_type, product_name, shade_name;

In [ ]:
import pandas as pd
import numpy as np
import colorsys

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

def rgb_to_hsl(r, g, b):
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return h * 360, l * 100, s * 100

df = all_colors.to_pandas() if hasattr(all_colors, 'to_pandas') else all_colors
df = df.copy()
df.columns = [col.lower() for col in df.columns]

df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)
df['hue'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[0])
df['lightness'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[1])

era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Eyeshadow, eyeliner, lip, blush, mascara, and nail shades',
    'revival': 'Bolder, more saturated color across every category',
}

def get_extreme_labels(era_df, n=5):
    extremes = set()
    if len(era_df) <= n * 2:
        return set(era_df['shade_name'])
    extremes.update(era_df.nsmallest(n, 'lightness')['shade_name'])
    extremes.update(era_df.nlargest(n, 'lightness')['shade_name'])
    extremes.update(era_df.nsmallest(3, 'hue')['shade_name'])
    extremes.update(era_df.nlargest(3, 'hue')['shade_name'])
    return extremes

def plot_both(layout='horizontal'):
    era_order = ['original_launch', 'revival']
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)

    for i, (axis, era) in enumerate(zip(axes.flat, era_order)):
        era_df = df[df['era'] == era].copy()
        colors = list(era_df['rgb'])
        axis.scatter(
            era_df['hue'], era_df['lightness'],
            s=100, alpha=0.9, c=colors,
            edgecolors='black', linewidths=0.4,
        )
        extremes = get_extreme_labels(era_df)
        for _, row in era_df.iterrows():
            if row['shade_name'] in extremes:
                axis.annotate(
                    row['shade_name'],
                    (row['hue'], row['lightness']),
                    textcoords='offset points', xytext=(5, 4),
                    fontsize=9, fontweight='normal', alpha=0.85,
                )
        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Hue (color family)', fontsize=11, fontweight='normal')
        axis.set_xlim(-10, 370)
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)
        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Lightness (%)', fontsize=11, fontweight='normal')

    if layout == 'horizontal':
        fig.suptitle('EVERY COLOR ACROSS BOTH COLLECTIONS (EXCL. COMPLEXION)',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('EVERY COLOR ACROSS BOTH COLLECTIONS (EXCL. COMPLEXION)',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_both('horizontal')
plot_both('vertical')

def plot_single_era(era, title, show_labels=True, transparent=False, no_edge=False):
    fig, ax = plt.subplots(figsize=(10, 7))
    if transparent:
        fig.patch.set_alpha(0)
        ax.patch.set_alpha(0)

    era_df = df[df['era'] == era].copy()
    colors = list(era_df['rgb'])
    ax.scatter(
        era_df['hue'], era_df['lightness'],
        s=120, alpha=0.9, c=colors,
        edgecolors='none' if no_edge else 'black',
        linewidths=0 if no_edge else 0.4,
    )

    if show_labels:
        extremes = get_extreme_labels(era_df)
        for _, row in era_df.iterrows():
            if row['shade_name'] in extremes:
                ax.annotate(
                    row['shade_name'],
                    (row['hue'], row['lightness']),
                    textcoords='offset points', xytext=(5, 4),
                    fontsize=9, fontweight='normal', alpha=0.85,
                )
        ax.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        ax.text(0.5, 1.01, era_annotations[era],
                transform=ax.transAxes, ha='center', fontsize=11,
                fontweight='normal', color='#444')
        ax.set_xlabel('Hue (color family)', fontsize=11, fontweight='normal')
        ax.set_ylabel('Lightness (%)', fontsize=11, fontweight='normal')
        fig.suptitle(title, fontsize=18, fontweight='medium', y=0.98)
    else:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)

    ax.set_xlim(-10, 370)
    ax.grid(alpha=0.15 if show_labels else 0)
    ax.tick_params(labelsize=9)
    plt.tight_layout()
    plt.show()

plot_single_era('original_launch', '2013 COLLECTION: ALL COLORS')
plot_single_era('revival', '2026 REVIVAL: ALL COLORS')
plot_single_era('revival', '', show_labels=False, transparent=True, no_edge=True)